In [15]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json
import re

import pandas as pd


# =========================================================
# 1. 설정
# =========================================================

# Jupyter Notebook을 실행한 현재 폴더
BASE_DIR = Path.cwd().resolve()

# 결과 저장 폴더
OUTPUT_DIR = BASE_DIR / "fixtures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Django 모델 경로
# 실제 앱 또는 모델명이 다르면 수정
MODEL_LABEL = "stores.store"

# Store 모델에 created_at, updated_at 필드가 있으면 True
# 해당 필드가 없다면 False로 변경
ADD_TIMESTAMP_FIELDS = True

# 옷가게만 추출
TARGET_CATEGORY_CODES = {
    "G20901",  # 남성 의류
    "G20902",  # 여성 의류
    "G20903",  # 유아용 의류
    "G20904",  # 한복
    "G20905",  # 기타 의류
}


print("현재 작업 폴더:", BASE_DIR)
print("결과 저장 폴더:", OUTPUT_DIR)

현재 작업 폴더: C:\Users\SSAFY\Downloads
결과 저장 폴더: C:\Users\SSAFY\Downloads\fixtures


In [16]:
# =========================================================
# 2. CSV와 Region fixture 자동 탐색
# =========================================================

def extract_yyyymm(path: Path) -> str:
    match = re.search(r"_(\d{6})(?:\.[^.]+)?$", path.name)
    return match.group(1) if match else "000000"


# Notebook과 같은 폴더에 있는 서울 상가 CSV 탐색
csv_candidates = list(
    BASE_DIR.glob(
        "소상공인시장진흥공단_상가(상권)정보_서울_*.csv"
    )
)

if not csv_candidates:
    raise FileNotFoundError(
        "현재 폴더에서 서울 상가 CSV 파일을 찾지 못했습니다.\n"
        f"현재 폴더: {BASE_DIR}"
    )

# 여러 파일이 있으면 기준 연월이 가장 최신인 파일 사용
CSV_PATH = max(
    csv_candidates,
    key=extract_yyyymm,
)

SNAPSHOT_YM = extract_yyyymm(CSV_PATH)


# Region fixture 탐색
# 현재 폴더 또는 fixtures 폴더에 두면 됨
region_candidates = (
    list(BASE_DIR.glob("region_fixture_*.json"))
    + list(OUTPUT_DIR.glob("region_fixture_*.json"))
)

if not region_candidates:
    raise FileNotFoundError(
        "region_fixture_*.json 파일을 찾지 못했습니다.\n"
        "기존 Region fixture를 Untitled.ipynb와 같은 폴더에 복사해주세요."
    )

REGION_FIXTURE_PATH = max(
    region_candidates,
    key=extract_yyyymm,
)


print("사용할 상가 CSV:", CSV_PATH.name)
print("사용할 Region fixture:", REGION_FIXTURE_PATH.name)
print("데이터 기준 연월:", SNAPSHOT_YM)

사용할 상가 CSV: 소상공인시장진흥공단_상가(상권)정보_서울_202603.csv
사용할 Region fixture: region_fixture_20260609_onlyseoul.json
데이터 기준 연월: 202603


In [17]:
# =========================================================
# 3. CSV 인코딩 확인
# =========================================================

def detect_encoding(csv_path: Path) -> str:
    candidate_encodings = [
        "utf-8-sig",
        "utf-8",
        "cp949",
        "euc-kr",
    ]

    for encoding in candidate_encodings:
        try:
            pd.read_csv(
                csv_path,
                encoding=encoding,
                dtype=str,
                nrows=5,
            )
            return encoding
        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"CSV 인코딩을 확인할 수 없습니다: {csv_path}"
    )


CSV_ENCODING = detect_encoding(CSV_PATH)

print("CSV 인코딩:", CSV_ENCODING)

CSV 인코딩: utf-8-sig


In [18]:
# =========================================================
# 4. 공통 데이터 정리 함수
# =========================================================

def clean_text(value) -> str:
    if value is None or pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value).strip(),
    )


def normalize_region_code(value) -> str:
    """
    1168010100.0 등의 값을 1168010100으로 정리
    """
    text = clean_text(value)

    if text.endswith(".0"):
        text = text[:-2]

    return re.sub(r"[^0-9]", "", text)


def normalize_region_name(value) -> str:
    """
    지역명 비교를 위해 공백을 정리
    """
    return re.sub(
        r"\s+",
        "",
        clean_text(value),
    )

In [19]:
# =========================================================
# 5. Region fixture를 PK 매핑 자료로 변환
# =========================================================

with open(
    REGION_FIXTURE_PATH,
    "r",
    encoding="utf-8",
) as file:
    region_fixture_data = json.load(file)


# Region 모델의 코드 필드명 후보
REGION_CODE_FIELD_CANDIDATES = [
    "region_code",
    "code",
    "administrative_code",
    "legal_dong_code",
    "bjd_code",
]


region_code_to_pk = {}
region_name_to_pk = {}


for fixture_object in region_fixture_data:
    region_pk = fixture_object["pk"]
    fields = fixture_object["fields"]

    # region fixture의 코드 필드명 자동 탐색
    region_code = ""

    for field_name in REGION_CODE_FIELD_CANDIDATES:
        if field_name in fields:
            region_code = normalize_region_code(
                fields[field_name]
            )

            if region_code:
                break

    if region_code:
        region_code_to_pk[region_code] = region_pk

    # 코드 매칭 실패에 대비해 이름 기준 매핑도 생성
    sido = normalize_region_name(
        fields.get("sido", "")
    )
    sigungu = normalize_region_name(
        fields.get("sigungu", "")
    )
    dong = normalize_region_name(
        fields.get("dong", "")
    )

    if sido and sigungu and dong:
        region_name_to_pk[
            (
                sido,
                sigungu,
                dong,
            )
        ] = region_pk


print(
    "Region 코드 매핑 수:",
    f"{len(region_code_to_pk):,}",
)

print(
    "Region 이름 매핑 수:",
    f"{len(region_name_to_pk):,}",
)

Region 코드 매핑 수: 1,110
Region 이름 매핑 수: 1,084


In [20]:
# =========================================================
# 6. CSV 컬럼 확인
# =========================================================

header_df = pd.read_csv(
    CSV_PATH,
    encoding=CSV_ENCODING,
    dtype=str,
    nrows=0,
)

available_columns = header_df.columns.tolist()


required_columns = [
    "상가업소번호",
    "상호명",
    "상권업종소분류코드",
    "상권업종소분류명",
    "시도명",
    "시군구명",
    "지번주소",
    "도로명주소",
    "경도",
    "위도",
]


optional_columns = [
    "지점명",
    "행정동코드",
    "행정동명",
    "법정동코드",
    "법정동명",
]


missing_columns = [
    column
    for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(
        "CSV에 필요한 컬럼이 없습니다.\n"
        f"누락 컬럼: {missing_columns}\n"
        f"실제 컬럼: {available_columns}"
    )


use_columns = required_columns + [
    column
    for column in optional_columns
    if column in available_columns
]


print("사용할 CSV 컬럼:")

for column in use_columns:
    print("-", column)

사용할 CSV 컬럼:
- 상가업소번호
- 상호명
- 상권업종소분류코드
- 상권업종소분류명
- 시도명
- 시군구명
- 지번주소
- 도로명주소
- 경도
- 위도
- 지점명
- 행정동코드
- 행정동명
- 법정동코드
- 법정동명


In [21]:
# =========================================================
# 7. 상가 데이터와 Region 연결 함수
# =========================================================

def find_region_pk(row):
    """
    1. 행정동코드
    2. 법정동코드
    3. 행정동 이름
    4. 법정동 이름

    순서로 Region PK를 찾는다.
    """

    # 1. 행정동코드로 매칭
    if "행정동코드" in row.index:
        administrative_code = normalize_region_code(
            row["행정동코드"]
        )

        if administrative_code in region_code_to_pk:
            return region_code_to_pk[administrative_code]

    # 2. 법정동코드로 매칭
    if "법정동코드" in row.index:
        legal_code = normalize_region_code(
            row["법정동코드"]
        )

        if legal_code in region_code_to_pk:
            return region_code_to_pk[legal_code]

    sido = normalize_region_name(
        row.get("시도명", "")
    )

    sigungu = normalize_region_name(
        row.get("시군구명", "")
    )

    # 3. 행정동명으로 매칭
    if "행정동명" in row.index:
        administrative_dong = normalize_region_name(
            row["행정동명"]
        )

        administrative_key = (
            sido,
            sigungu,
            administrative_dong,
        )

        if administrative_key in region_name_to_pk:
            return region_name_to_pk[
                administrative_key
            ]

    # 4. 법정동명으로 매칭
    if "법정동명" in row.index:
        legal_dong = normalize_region_name(
            row["법정동명"]
        )

        legal_key = (
            sido,
            sigungu,
            legal_dong,
        )

        if legal_key in region_name_to_pk:
            return region_name_to_pk[legal_key]

    return None

In [22]:
# =========================================================
# 8. 대용량 CSV에서 옷가게만 추출
# =========================================================

filtered_chunks = []
unmatched_chunks = []

total_count = 0
clothing_count = 0
coordinate_missing_count = 0


for chunk_number, chunk in enumerate(
    pd.read_csv(
        CSV_PATH,
        encoding=CSV_ENCODING,
        dtype=str,
        usecols=use_columns,
        chunksize=100_000,
        low_memory=False,
    ),
    start=1,
):
    total_count += len(chunk)

    chunk = chunk.fillna("")

    for column in chunk.columns:
        chunk[column] = (
            chunk[column]
            .astype(str)
            .str.strip()
        )

    category_code = chunk[
        "상권업종소분류코드"
    ]

    category_name = chunk[
        "상권업종소분류명"
    ]

    # 1. 알려진 의류 업종 코드
    code_mask = category_code.isin(
        TARGET_CATEGORY_CODES
    )

    # 2. 업종 코드 변경 가능성에 대비한 업종명 기준 필터
    # '의류 소매업', '한복 소매업', '중고 의류 소매업' 등
    name_mask = (
        category_name.str.contains(
            r"의류|한복",
            na=False,
            regex=True,
        )
        & category_name.str.contains(
            r"소매|판매",
            na=False,
            regex=True,
        )
    )

    # 제조·도매·수선·세탁·대여 업종 제외
    exclusion_mask = category_name.str.contains(
        r"제조|도매|수선|세탁|대여|임대|원단|직물",
        na=False,
        regex=True,
    )

    clothing_mask = (
        code_mask | name_mask
    ) & ~exclusion_mask

    clothing_chunk = chunk.loc[
        clothing_mask
    ].copy()

    clothing_count += len(clothing_chunk)

    if clothing_chunk.empty:
        print(
            f"{chunk_number}번 청크: 옷가게 0개"
        )
        continue

    # 좌표를 숫자로 변환
    clothing_chunk["longitude_number"] = (
        pd.to_numeric(
            clothing_chunk["경도"],
            errors="coerce",
        )
    )

    clothing_chunk["latitude_number"] = (
        pd.to_numeric(
            clothing_chunk["위도"],
            errors="coerce",
        )
    )

    invalid_coordinate_mask = (
        clothing_chunk["longitude_number"].isna()
        | clothing_chunk["latitude_number"].isna()
    )

    coordinate_missing_count += int(
        invalid_coordinate_mask.sum()
    )

    clothing_chunk = clothing_chunk.loc[
        ~invalid_coordinate_mask
    ].copy()

    # 필수 식별값 없는 데이터 제외
    clothing_chunk = clothing_chunk[
        clothing_chunk["상가업소번호"].ne("")
        & clothing_chunk["상호명"].ne("")
    ].copy()

    # 기존 Region fixture의 PK 연결
    clothing_chunk["region_pk"] = (
        clothing_chunk.apply(
            find_region_pk,
            axis=1,
        )
    )

    unmatched_mask = (
        clothing_chunk["region_pk"].isna()
    )

    if unmatched_mask.any():
        unmatched_columns = [
            column
            for column in [
                "상가업소번호",
                "상호명",
                "시도명",
                "시군구명",
                "행정동코드",
                "행정동명",
                "법정동코드",
                "법정동명",
                "도로명주소",
            ]
            if column in clothing_chunk.columns
        ]

        unmatched_chunks.append(
            clothing_chunk.loc[
                unmatched_mask,
                unmatched_columns,
            ].copy()
        )

    # Region FK를 찾은 데이터만 fixture에 사용
    matched_chunk = clothing_chunk.loc[
        ~unmatched_mask
    ].copy()

    if not matched_chunk.empty:
        filtered_chunks.append(
            matched_chunk
        )

    print(
        f"{chunk_number}번 청크: "
        f"전체 {len(chunk):,}개 / "
        f"옷가게 {len(clothing_chunk):,}개 / "
        f"Region 연결 {len(matched_chunk):,}개"
    )


if not filtered_chunks:
    raise ValueError(
        "Region과 연결된 옷가게 데이터가 없습니다.\n"
        "Region fixture의 코드 및 지역명을 확인해주세요."
    )


stores_df = pd.concat(
    filtered_chunks,
    ignore_index=True,
)


print()
print("전체 상가 수:", f"{total_count:,}")
print("옷가게 분류 수:", f"{clothing_count:,}")
print(
    "좌표 누락 제외 수:",
    f"{coordinate_missing_count:,}",
)
print(
    "Region 연결 완료 수:",
    f"{len(stores_df):,}",
)

1번 청크: 전체 100,000개 / 옷가게 4,600개 / Region 연결 4,600개
2번 청크: 전체 100,000개 / 옷가게 4,556개 / Region 연결 4,556개
3번 청크: 전체 100,000개 / 옷가게 4,582개 / Region 연결 4,582개
4번 청크: 전체 100,000개 / 옷가게 3,333개 / Region 연결 3,333개
5번 청크: 전체 100,000개 / 옷가게 1,776개 / Region 연결 1,776개
6번 청크: 전체 37,489개 / 옷가게 1,380개 / Region 연결 1,380개

전체 상가 수: 537,489
옷가게 분류 수: 20,227
좌표 누락 제외 수: 0
Region 연결 완료 수: 20,227


In [23]:
# =========================================================
# 9. 중복 제거 및 정렬
# =========================================================

before_deduplication = len(stores_df)

stores_df = stores_df.drop_duplicates(
    subset=["상가업소번호"],
    keep="first",
)

stores_df["region_pk"] = (
    stores_df["region_pk"].astype(int)
)

stores_df = stores_df.sort_values(
    by=[
        "region_pk",
        "상호명",
        "상가업소번호",
    ]
).reset_index(drop=True)


print(
    "중복 제거 수:",
    f"{before_deduplication - len(stores_df):,}",
)

print(
    "최종 옷가게 수:",
    f"{len(stores_df):,}",
)

중복 제거 수: 0
최종 옷가게 수: 20,227


In [24]:
# =========================================================
# 10. Django fixture 형식으로 변환
# =========================================================

KST = timezone(
    timedelta(hours=9)
)

generated_at = datetime.now(
    KST
).isoformat(
    timespec="seconds"
)


fixture_data = []


for fixture_pk, row in enumerate(
    stores_df.to_dict(orient="records"),
    start=1,
):
    fields = {
        # Store 모델의 필드명과 반드시 같아야 함
        "external_store_id": clean_text(
            row["상가업소번호"]
        ),
        "name": clean_text(
            row["상호명"]
        ),
        "branch_name": clean_text(
            row.get("지점명", "")
        ),
        "category_code": clean_text(
            row["상권업종소분류코드"]
        ),
        "category_name": clean_text(
            row["상권업종소분류명"]
        ),

        # ForeignKey 필드는 region_id가 아니라 region
        "region": int(
            row["region_pk"]
        ),

        "jibun_address": clean_text(
            row["지번주소"]
        ),
        "road_address": clean_text(
            row["도로명주소"]
        ),
        "longitude": round(
            float(row["longitude_number"]),
            7,
        ),
        "latitude": round(
            float(row["latitude_number"]),
            7,
        ),
        "data_source": (
            "소상공인시장진흥공단_상가(상권)정보"
        ),
        "is_active": True,
    }

    if ADD_TIMESTAMP_FIELDS:
        fields["created_at"] = generated_at
        fields["updated_at"] = generated_at

    fixture_data.append(
        {
            "model": MODEL_LABEL,
            "pk": fixture_pk,
            "fields": fields,
        }
    )


print(
    "Fixture 변환 완료:",
    f"{len(fixture_data):,}개",
)

Fixture 변환 완료: 20,227개


In [25]:
# =========================================================
# 11. JSON fixture 저장
# =========================================================

OUTPUT_JSON_PATH = (
    OUTPUT_DIR
    / f"store_fixture_{SNAPSHOT_YM}.json"
)


with open(
    OUTPUT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        fixture_data,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )


print("JSON 저장 완료")
print("저장 경로:", OUTPUT_JSON_PATH)
print(
    "저장된 옷가게 수:",
    f"{len(fixture_data):,}",
)

JSON 저장 완료
저장 경로: C:\Users\SSAFY\Downloads\fixtures\store_fixture_202603.json
저장된 옷가게 수: 20,227


In [26]:
# =========================================================
# 12. Region 매칭 실패 데이터 저장
# =========================================================

if unmatched_chunks:
    unmatched_df = pd.concat(
        unmatched_chunks,
        ignore_index=True,
    )

    unmatched_df = unmatched_df.drop_duplicates(
        subset=["상가업소번호"],
        keep="first",
    )

    UNMATCHED_CSV_PATH = (
        OUTPUT_DIR
        / f"store_region_unmatched_{SNAPSHOT_YM}.csv"
    )

    unmatched_df.to_csv(
        UNMATCHED_CSV_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print()
    print(
        "Region 매칭 실패 수:",
        f"{len(unmatched_df):,}",
    )
    print(
        "매칭 실패 확인 파일:",
        UNMATCHED_CSV_PATH,
    )
else:
    print()
    print("Region 매칭 실패 데이터 없음")


Region 매칭 실패 데이터 없음


In [29]:
# =========================================================
# 13. 결과 검증
# =========================================================

category_summary = (
    stores_df
    .groupby(
        [
            "상권업종소분류코드",
            "상권업종소분류명",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="store_count"
    )
    .sort_values(
        "store_count",
        ascending=False,
    )
)

display(category_summary)


preview_columns = [
    column
    for column in [
        "상가업소번호",
        "상호명",
        "지점명",
        "상권업종소분류명",
        "시군구명",
        "행정동명",
        "법정동명",
        "도로명주소",
        "region_pk",
        "longitude_number",
        "latitude_number",
    ]
    if column in stores_df.columns
]

display(
    stores_df[
        preview_columns
    ].tail(40)
)

,상권업종소분류코드,상권업종소분류명,store_count
4,G20905,기타 의류 소매업,11039
1,G20902,여성 의류 소매업,6524
0,G20901,남성 의류 소매업,1695
2,G20903,유아용 의류 소매업,738
3,G20904,한복 소매업,231


,상가업소번호,상호명,지점명,상권업종소분류명,시군구명,행정동명,법정동명,도로명주소,region_pk,longitude_number,latitude_number
20187,MA010120220802284323,더써빗,,기타 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉로159길 19,1073,127.043097,37.674306
20188,MA010120220803978889,도봉산잡화점,,기타 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉산1길 62-5,1073,127.044928,37.684459
20189,MA0106202509A0609354,레드페이스도1,봉산점,기타 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉산길 65-5,1073,127.037855,37.686345
20190,MA010120220803085775,루렌샤,,기타 의류 소매업,도봉구,도봉2동,도봉동,서울특별시 도봉구 도봉로180가길 96,1073,127.046506,37.687792
20191,MA0106202511A0710657,마운티아도,봉산점,기타 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉산길 65-5,1073,127.037855,37.686345
20192,MA010120220804000870,마이캐빈주,,기타 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉산길 65-5,1073,127.037855,37.686345
20193,MA010120220806656228,모자나라,,기타 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉산4나길 5,1073,127.042731,37.688325
20194,MA010120220804003099,미래베이직도봉산지점,코리아,기타 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉산4나길 6-3,1073,127.042992,37.688136
20195,MA010120220806541629,미미네옷방,,여성 의류 소매업,도봉구,도봉1동,도봉동,서울특별시 도봉구 도봉로153길 56,1073,127.040284,37.670045
20196,MA010120220803944442,밀라노토탈패션,,여성 의류 소매업,도봉구,도봉2동,도봉동,서울특별시 도봉구 도봉로180나길 41,1073,127.049894,37.684349
